# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
pop_rows = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]
print(f"Count: {len(pop_rows)}")
print(pop_rows)

Count: 3
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

**Strawberry Pop-Tart SKUs/Names**: PT-12 (pop-tarts strawberry), SKU#459812 (Strawberry Pop-Tart), and POPTART-STRAW (Pop-Tarts Strawberry).

**Pop-Tart rows that are a different product:** Any other Pop-Tart variants (such as blueberry, brown sugar, or hot fudge sundae) must stay separate because they represent distinct physical inventory items and SKU variants.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [3]:
sku_mapping = {
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12'
}
cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [4]:
print("Before:", cat['category'].value_counts(dropna=False))

cat['category'] = cat['category'].str.strip().str.title()

print("\nAfter:", cat['category'].value_counts(dropna=False))

Before: category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

After: category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [5]:
import re

def normalize_name(name):
    if not isinstance(name, str):
        return ""
    cleaned = re.sub(r'[^\w\s]', '', name.lower())
    return ' '.join(sorted(cleaned.split()))

cat['normalized_name'] = cat['product_name'].apply(normalize_name)

fragmented = cat.groupby('normalized_name')['sku'].unique()
fragmented = fragmented[fragmented.apply(len) > 1]
print(fragmented)

normalized_name
poptarts strawberry    [PT-12, POPTART-STRAW]
Name: sku, dtype: object


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [6]:
assert cat['canonical_sku'].isnull().sum() == 0, "Mapping produced null values!"
assert len(cat) == len(cat['sku']), "Row count changed during mapping!"
print("Validation passed: Mapping is safe.")

Validation passed: Mapping is safe.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a. Two additional product families that typically appear fragmented in retail catalogs are packaged beverages which get split across abbreviations like Coke 12pk, Coca-Cola 12 oz cans, and CC 12 pack—and laundry detergents, which vary by scent and volume descriptors. I would group them by extracting base brand/product tokens using regex patterns for volume/size and standardizing flavor/scent descriptors.

b. Consolidation becomes a business decision when two SKUs have slightly different descriptions (ex: a packaging update, a slight formula tweak, or a bundled promo pack vs. a standalone item) where it is unclear whether inventory and sales history should be merged. An example in a catalog like this would be deciding whether a 12-count box and a 12-count value pack share a canonical SKU. You would want to consult the Inventory Manager or Category Merchandiser to determine if they are tracked interchangeably in supply chain planning.